In [3]:
!pip install -U langchain langchain_core langchain_classic langchain_community langchain_google_genai

  Using cached langchain-1.2.17-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Suc

In [5]:
!pip install HuggingFace  crewai  google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [27]:
import os
import getpass
os.environ["GOOGLE_API_KEY"]=getpass.getpass("enter the api key")

enter the api key··········


In [64]:

#llm
from crewai import LLM
from langchain_google_genai import ChatGoogleGenerativeAI
llm=LLM(
    model="gemini-2.5-flash",
    temperature=0.3
)


In [65]:
from crewai import Agent
sql_generator=Agent(
    role='SQl Generator',
    goal='Convert the user requirement into the accurate and logically as well syntax correct SQL queries',
    backstory='Your are an expert data analyst with 5+ years of experience of analyzing and understanding data in several domains and who writes the effective SQL queries',
    llm=llm,
    verbose=True
)

sql_optimizer = Agent(
    role="SQL Optimizer",
    goal="Optimize SQL queries for performance and readability",
    backstory="You are a database performance expert skilled in query optimization.",
    llm=llm,
    verbose=True
)

sql_explainer = Agent(
    role="SQL Explainer",
    goal="Explain SQL queries in simple and understandable terms",
    backstory="You simplify complex SQL for beginners.",
    llm=llm,
    verbose=True
)

In [66]:
from crewai import Task

task_generate = Task(
    description="""
    Convert the user query into SQL.

    User Query: {user_query}

    Table: transactions(user_id, amount, date)

    Output only SQL query.
    """,
    agent=sql_generator,
    expected_output="A SQL query that accurately converts the user query into a valid and runnable SQL statement based on the provided table schema."
)

In [67]:
task_optimize = Task(
    description="""
    Optimize the following SQL query:

    {task_generate.output}

    Improve performance and readability.
    Also suggest indexes.
    """,
    agent=sql_optimizer,
    expected_output="An optimized SQL query with improved performance and readability, including suggestions for relevant indexes."
)

task_explain = Task(
    description="""
    Explain this SQL query in simple terms:

    {task_generate.output}
    """,
    agent=sql_explainer,
    expected_output="A simple and easy-to-understand explanation of the provided SQL query."
)

In [68]:
from crewai import Crew

crew = Crew(
    agents=[sql_generator, sql_optimizer, sql_explainer],
    tasks=[task_generate, task_optimize, task_explain],
    verbose=True
)

In [69]:
result = crew.kickoff(
    inputs={"user_query": "Get last 6 months user-wise monthly average revenue"}
)

print(result)

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 0cda900f-7018-4371-b973-fe60be4e0add                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                                              │
│  Agent: SQl Generator                                                        │
│                                                                              │
│  Task:                     

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
╭──────────────────────────────── ❌ LLM Error ────────────────────────────────╮
│                                                                              │
│  LLM Call Failed                                                             │
│  Error: Google Gemini API error: 503 - This model is currently experiencing  │
│  high demand. Spikes in demand are usually temporary. Please try again       │
│  later.                                                 

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
╭──────────────────────────────── ❌ LLM Error ────────────────────────────────╮
│                                                                              │
│  LLM Call Failed                                                             │
│  Error: Google Gemini API error: 503 - This model is currently experiencing  │
│  high demand. Spikes in demand are usually temporary. Please try again       │
│  later.                                                 

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
╭──────────────────────────────── ❌ LLM Error ────────────────────────────────╮
│                                                                              │
│  LLM Call Failed                                                             │
│  Error: Google Gemini API error: 503 - This model is currently experiencing  │
│  high demand. Spikes in demand are usually temporary. Please try again       │
│  later.                                                 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}